In [1]:
# rank_aware_10_experiments_final.py
# Corrected 10-experiment pipeline for rank-aware polynomial/eigenvalue validation.

from __future__ import annotations

import os
import time
import warnings
from itertools import combinations
from typing import Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy import stats
except Exception:
    stats = None

OUTPUT_DIR = "rank_aware_10_experiment_outputs"
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

EPS = 1e-300
FAIL_TOL = 1e-8


# ============================================================
# Basic polynomial utilities
# ============================================================

def poly_eval(coeffs_desc, x):
    coeffs_desc = np.asarray(coeffs_desc, dtype=complex)
    return np.polyval(coeffs_desc, x)


def poly_derivative(coeffs_desc):
    coeffs_desc = np.asarray(coeffs_desc, dtype=complex)
    n = len(coeffs_desc) - 1
    if n <= 0:
        return np.array([0.0], dtype=complex)
    return np.array([coeffs_desc[i] * (n - i) for i in range(n)], dtype=complex)


def stable_power_sum(abs_x, n):
    """Safely compute sum_{j=0}^n |x|^j."""
    abs_x = np.asarray(abs_x, dtype=float)
    S = np.zeros_like(abs_x, dtype=float)
    close = np.abs(abs_x - 1.0) < 1e-12
    S[close] = n + 1
    mask = ~close
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        S[mask] = (abs_x[mask] ** (n + 1) - 1.0) / (abs_x[mask] - 1.0)
    S[~np.isfinite(S)] = np.inf
    return np.maximum(S, EPS)


def backward_error(coeffs_desc, roots):
    """Scale-invariant backward error."""
    coeffs_desc = np.asarray(coeffs_desc, dtype=complex)
    roots = np.asarray(roots, dtype=complex)
    if roots.size == 0:
        return np.array([np.inf])
    n = len(coeffs_desc) - 1
    coeff_norm = max(np.sum(np.abs(coeffs_desc)), EPS)
    fx = np.abs(poly_eval(coeffs_desc, roots))
    scale = coeff_norm * stable_power_sum(np.abs(roots), n)
    eta = fx / np.maximum(scale, EPS)
    eta[~np.isfinite(eta)] = np.inf
    return eta


def max_backward_error(coeffs_desc, roots):
    return float(np.nanmax(backward_error(coeffs_desc, roots)))


def raw_root_residual(coeffs_desc, roots):
    """Appendix-only raw residual max |f(x)|."""
    roots = np.asarray(roots, dtype=complex)
    if roots.size == 0:
        return np.inf
    vals = np.abs(poly_eval(coeffs_desc, roots))
    vals[~np.isfinite(vals)] = np.inf
    return float(np.nanmax(vals))


def normalized_root_residual(coeffs_desc, roots):
    """Main residual metric, same normalization as backward error."""
    return max_backward_error(coeffs_desc, roots)


def newton_refine(coeffs_desc, roots, max_iter=25, tol=1e-12):
    coeffs_desc = np.asarray(coeffs_desc, dtype=complex)
    roots = np.asarray(roots, dtype=complex).copy()
    if roots.size == 0:
        return roots
    dcoeffs = poly_derivative(coeffs_desc)
    for _ in range(max_iter):
        fx = poly_eval(coeffs_desc, roots)
        dfx = poly_eval(dcoeffs, roots)
        step = np.zeros_like(roots, dtype=complex)
        valid = np.abs(dfx) > 1e-14
        step[valid] = fx[valid] / dfx[valid]
        step[~np.isfinite(step)] = 0.0
        new_roots = roots - step
        if np.nanmax(np.abs(new_roots - roots)) < tol:
            roots = new_roots
            break
        roots = new_roots
    return roots


# ============================================================
# Structured anti-diagonal representation
# ============================================================

def anti_diagonal_poly_from_matrix(A, degree=None):
    A = np.asarray(A, dtype=float)
    m = A.shape[0]
    d = m - 1
    max_degree = 2 * d
    coeffs_desc = np.zeros(max_degree + 1, dtype=float)
    for i in range(m):
        for j in range(m):
            power = 2 * d - (i + j)
            idx = max_degree - power
            coeffs_desc[idx] += A[i, j]
    if degree is not None:
        coeffs_desc = coeffs_desc[-(degree + 1):]
    return coeffs_desc


def canonical_A_from_poly(coeffs_desc):
    coeffs_desc = np.asarray(coeffs_desc, dtype=float)
    n = len(coeffs_desc) - 1
    d = int(np.ceil(n / 2))
    m = d + 1
    A = np.zeros((m, m), dtype=float)
    for idx, coeff in enumerate(coeffs_desc):
        power = n - idx
        anti_sum = 2 * d - power
        pairs = [(i, j) for i in range(m) for j in range(m) if i + j == anti_sum]
        if not pairs:
            continue
        share = coeff / len(pairs)
        for i, j in pairs:
            A[i, j] += share
    return A


def coefficient_mismatch(f_true, f_hat):
    f_true = np.asarray(f_true, dtype=float)
    f_hat = np.asarray(f_hat, dtype=float)
    max_len = max(len(f_true), len(f_hat))
    a = np.pad(f_true, (max_len - len(f_true), 0))
    b = np.pad(f_hat, (max_len - len(f_hat), 0))
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a), EPS))


# ============================================================
# Rank diagnostics
# ============================================================

def singular_values(A):
    return np.linalg.svd(np.asarray(A, dtype=float), compute_uv=False)


def numerical_rank(A, tol=1e-10):
    return int(np.sum(singular_values(A) > tol))


def sigma_k(A, k):
    s = singular_values(A)
    if k <= 0 or k > len(s):
        return 0.0
    return float(s[k - 1])


def rho_sigma_k(A, k):
    s = singular_values(A)
    if k >= len(s):
        return 0.0
    return float(s[k] / max(s[0], EPS))


def max_abs_minor(A, size):
    A = np.asarray(A, dtype=float)
    m = A.shape[0]
    if size > m:
        return 0.0
    if m > 12:
        s = singular_values(A)
        return float(np.prod(s[size - 1:])) if size - 1 < len(s) else 0.0
    best = 0.0
    for I in combinations(range(m), size):
        for J in combinations(range(m), size):
            best = max(best, abs(np.linalg.det(A[np.ix_(I, J)])))
    return float(best)


# ============================================================
# Data generation and approximations
# ============================================================

def generate_exact_rank2_polynomial(n, rng):
    d = int(np.ceil(n / 2))
    m = d + 1
    v1 = rng.normal(size=m)
    v2 = rng.normal(size=m)
    if n % 2 == 1:
        v2[0] = v1[0]
    A = np.outer(v1, v1) - np.outer(v2, v2)
    coeffs = anti_diagonal_poly_from_matrix(A, degree=n)
    coeffs = coeffs / max(np.linalg.norm(coeffs), EPS)
    return coeffs, A, v1, v2


def exact_rank2_roots(v1, v2):
    return np.concatenate([np.roots(v1 + v2), np.roots(v1 - v2)])


def spectral_rank_k(A, k):
    evals, evecs = np.linalg.eigh(np.asarray(A, dtype=float))
    idx = np.argsort(np.abs(evals))[::-1][:k]
    U = evecs[:, idx]
    L = evals[idx]
    return (U * L) @ U.T


def sketch_qr_rank_k(A, k, oversample=10, seed=0):
    rng = np.random.default_rng(seed)
    A = np.asarray(A, dtype=float)
    m = A.shape[0]
    s = min(m, k + oversample)
    Omega = rng.normal(size=(m, s))
    start = time.perf_counter()
    Y = A @ Omega
    Q, _ = np.linalg.qr(Y, mode="reduced")
    B = Q.T @ A @ Q
    evals, evecs = np.linalg.eigh(B)
    idx = np.argsort(np.abs(evals))[::-1][:k]
    Uk = evecs[:, idx]
    Lk = evals[idx]
    Z = Q @ Uk
    Ak = (Z * Lk) @ Z.T
    return Ak, time.perf_counter() - start


# ============================================================
# Solvers
# ============================================================

def companion_solver(coeffs_desc):
    coeffs_desc = np.asarray(coeffs_desc, dtype=complex)
    if len(coeffs_desc) <= 1 or abs(coeffs_desc[0]) < EPS:
        return np.array([], dtype=complex), 0.0
    start = time.perf_counter()
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        roots = np.roots(coeffs_desc)
    runtime = time.perf_counter() - start
    roots = roots[np.isfinite(roots)]
    return roots, runtime


def durand_kerner(coeffs_desc, max_iter=200, tol=1e-12, max_degree=250):
    coeffs_desc = np.asarray(coeffs_desc, dtype=complex)
    n = len(coeffs_desc) - 1
    if n <= 0:
        return np.array([]), 0.0
    if n > max_degree or abs(coeffs_desc[0]) < EPS:
        return np.array([], dtype=complex), np.nan
    coeffs_desc = coeffs_desc / coeffs_desc[0]
    radius = 1.0 + np.max(np.abs(coeffs_desc[1:]))
    angles = 2.0 * np.pi * np.arange(n) / n
    roots = radius * np.exp(1j * angles)
    start = time.perf_counter()
    for _ in range(max_iter):
        old = roots.copy()
        for i in range(n):
            denom = np.prod(roots[i] - np.delete(roots, i))
            if abs(denom) > EPS:
                roots[i] -= poly_eval(coeffs_desc, roots[i]) / denom
        if np.max(np.abs(roots - old)) < tol:
            break
    roots = roots[np.isfinite(roots)]
    return roots, time.perf_counter() - start


def aberth_ehrlich(coeffs_desc, max_iter=100, tol=1e-12, max_degree=500):
    coeffs_desc = np.asarray(coeffs_desc, dtype=complex)
    n = len(coeffs_desc) - 1
    if n <= 0:
        return np.array([]), 0.0
    if n > max_degree:
        return np.array([], dtype=complex), np.nan
    roots, _ = companion_solver(coeffs_desc)
    if len(roots) != n:
        return np.array([], dtype=complex), np.nan
    roots = roots + 1e-8 * np.exp(2j * np.pi * np.arange(n) / n)
    dcoeffs = poly_derivative(coeffs_desc)
    start = time.perf_counter()
    for _ in range(max_iter):
        old = roots.copy()
        for i in range(n):
            fz = poly_eval(coeffs_desc, roots[i])
            dfz = poly_eval(dcoeffs, roots[i])
            if abs(dfz) < EPS:
                continue
            newton = fz / dfz
            diff = roots[i] - np.delete(roots, i)
            diff = diff[np.abs(diff) > EPS]
            correction_sum = np.sum(1.0 / diff) if diff.size else 0.0
            denom = 1.0 - newton * correction_sum
            if abs(denom) > EPS:
                roots[i] -= newton / denom
        if np.max(np.abs(roots - old)) < tol:
            break
    roots = roots[np.isfinite(roots)]
    return roots, time.perf_counter() - start


# ============================================================
# Experiments 1--10
# ============================================================

def experiment_1_exact_rank2(degrees=(5, 7, 20, 50), trials=20, seed=1):
    rng = np.random.default_rng(seed)
    rows = []
    for n in degrees:
        for t in range(trials):
            f, A, v1, v2 = generate_exact_rank2_polynomial(n, rng)
            roots = exact_rank2_roots(v1, v2)[:n]
            roots_ref = newton_refine(f, roots)
            rows.append({
                "degree_n": int(n), "trial": int(t), "m": int(A.shape[0]),
                "rank_A": int(numerical_rank(A)), "sigma3_A": sigma_k(A, 3),
                "Delta3_A": max_abs_minor(A, 3),
                "eta_max": max_backward_error(f, roots),
                "eta_max_refined": max_backward_error(f, roots_ref),
                "normalized_root_residual": normalized_root_residual(f, roots_ref),
                "raw_root_residual_appendix": raw_root_residual(f, roots_ref),
            })
    return pd.DataFrame(rows)


def experiment_2_statistical_robustness(degrees=(20, 50, 100, 200), trials=100, seed=2):
    df = experiment_1_exact_rank2(degrees=degrees, trials=trials, seed=seed)
    rows = []
    for degree, group in df.groupby("degree_n"):
        eta = group["eta_max_refined"].to_numpy(dtype=float)
        finite_eta = eta[np.isfinite(eta)]
        rows.append({
            "degree_n": int(degree), "trials": int(len(eta)),
            "eta_mean": float(np.mean(finite_eta)) if finite_eta.size else np.inf,
            "eta_sd": float(np.std(finite_eta, ddof=1)) if finite_eta.size > 1 else np.nan,
            "eta_median": float(np.median(finite_eta)) if finite_eta.size else np.inf,
            "eta_p95": float(np.percentile(finite_eta, 95)) if finite_eta.size else np.inf,
            "eta_worst_finite": float(np.max(finite_eta)) if finite_eta.size else np.inf,
            "failure_rate": float(np.mean(~np.isfinite(eta) | (eta > FAIL_TOL))),
            "sigma3_median": float(np.median(group["sigma3_A"])),
        })
    return pd.DataFrame(rows)


def experiment_3_perturbation(degree=50, noise_levels=(1e-12, 1e-10, 1e-8, 1e-6, 1e-4), trials=30, seed=3):
    rng = np.random.default_rng(seed)
    rows = []
    for eps in noise_levels:
        for t in range(trials):
            f, A, v1, v2 = generate_exact_rank2_polynomial(degree, rng)
            E = rng.normal(size=A.shape)
            E = (E + E.T) / 2.0
            E = E / max(np.linalg.norm(E, 2), EPS)
            A_eps = A + eps * np.linalg.norm(A, 2) * E
            f_eps = anti_diagonal_poly_from_matrix(A_eps, degree=degree)
            f_eps = f_eps / max(np.linalg.norm(f_eps), EPS)
            roots_eps, runtime = companion_solver(f_eps)
            roots_eps_ref = newton_refine(f_eps, roots_eps)
            roots_ref0 = exact_rank2_roots(v1, v2)[:degree]
            drift = np.nan
            if len(roots_eps_ref) == len(roots_ref0):
                drift = float(np.median(np.abs(np.sort_complex(roots_eps_ref) - np.sort_complex(roots_ref0))))
            rho2 = rho_sigma_k(A_eps, 2)
            rows.append({
                "degree_n": int(degree), "epsilon": float(eps), "trial": int(t),
                "rho_sigma_2": rho2, "sigma3_A": sigma_k(A_eps, 3),
                "Delta3_proxy": sigma_k(A_eps, 1) * sigma_k(A_eps, 2) * sigma_k(A_eps, 3),
                "eta_max_refined": max_backward_error(f_eps, roots_eps_ref),
                "normalized_root_residual": normalized_root_residual(f_eps, roots_eps_ref),
                "root_drift_median": drift, "runtime_s": runtime,
                "detect_rank2": bool(rho2 < 1e-6),
            })
    return pd.DataFrame(rows)


def experiment_4_rankk_surrogate(degrees=(100, 200, 400), ks=(2, 5, 10, 20, 40), seed=4):
    rng = np.random.default_rng(seed)
    rows = []
    for n in degrees:
        f = rng.normal(size=n + 1)
        f = f / max(np.linalg.norm(f), EPS)
        A = canonical_A_from_poly(f)
        roots_comp, _ = companion_solver(f)
        roots_comp_ref = newton_refine(f, roots_comp)
        comp_eta = max_backward_error(f, roots_comp_ref)
        for k in ks:
            if k >= A.shape[0]:
                continue
            Ak, sketch_time = sketch_qr_rank_k(A, k, oversample=10, seed=seed + n + k)
            fhat = anti_diagonal_poly_from_matrix(Ak, degree=n)
            roots_hat, root_time = companion_solver(fhat)
            roots_ref = newton_refine(f, roots_hat)
            eta_ref = max_backward_error(f, roots_ref)
            selected_eta = eta_ref
            selected_method = "rank-k surrogate"
            if eta_ref > 10 * comp_eta and comp_eta < FAIL_TOL:
                selected_eta = comp_eta
                selected_method = "companion fallback"
            rows.append({
                "degree_n": int(n), "m": int(A.shape[0]), "k": int(k),
                "rho_sigma_k": rho_sigma_k(A, k), "rel_coeff_error": coefficient_mismatch(f, fhat),
                "eta_max": max_backward_error(f, roots_hat), "eta_max_refined": eta_ref,
                "selected_eta": selected_eta, "selected_method": selected_method,
                "normalized_root_residual": normalized_root_residual(f, roots_ref),
                "sketch_time_s": sketch_time, "root_time_s": root_time,
            })
    return pd.DataFrame(rows)


def experiment_5_sketch_qr_scalability(degrees=(100, 200, 400, 800, 1000), k=20, seed=5):
    rng = np.random.default_rng(seed)
    rows = []
    for n in degrees:
        f = rng.normal(size=n + 1)
        f = f / max(np.linalg.norm(f), EPS)
        A = canonical_A_from_poly(f)
        kk = int(min(k, A.shape[0] - 1))
        start = time.perf_counter()
        Afull = spectral_rank_k(A, kk)
        full_time = time.perf_counter() - start
        Asketch, sketch_time = sketch_qr_rank_k(A, kk, oversample=10, seed=seed + n)
        f_full = anti_diagonal_poly_from_matrix(Afull, degree=n)
        f_sketch = anti_diagonal_poly_from_matrix(Asketch, degree=n)
        rows.append({
            "degree_n": int(n), "m": int(A.shape[0]), "k": int(kk),
            "full_eig_time_s": full_time, "sketch_qr_time_s": sketch_time,
            "full_coeff_error": coefficient_mismatch(f, f_full),
            "sketch_coeff_error": coefficient_mismatch(f, f_sketch),
            "rho_sigma_k": rho_sigma_k(A, kk), "memory_A_MB": A.nbytes / 1e6,
        })
    return pd.DataFrame(rows)


def wilkinson_coeffs(n):
    return np.poly(np.arange(1, n + 1)).astype(float)


def chebyshev_coeffs(n):
    from numpy.polynomial.chebyshev import Chebyshev
    from numpy.polynomial import Polynomial
    return Chebyshev.basis(n).convert(kind=Polynomial).coef[::-1].astype(float)


def experiment_6_benchmark_polynomials(degrees=(20, 40, 60, 80, 100), seed=6):
    rows = []
    for family in ["Wilkinson", "Chebyshev"]:
        for n in degrees:
            if family == "Wilkinson" and n > 80:
                continue
            f = wilkinson_coeffs(n) if family == "Wilkinson" else chebyshev_coeffs(n)
            f = np.asarray(f, dtype=float)
            f = f / max(np.linalg.norm(f), EPS)
            A = canonical_A_from_poly(f)
            roots_comp, time_comp = companion_solver(f)
            roots_comp_ref = newton_refine(f, roots_comp)
            companion_eta = max_backward_error(f, roots_comp_ref)
            k = int(min(20, A.shape[0] - 1))
            Ak, time_sketch = sketch_qr_rank_k(A, k, seed=seed + n)
            fhat = anti_diagonal_poly_from_matrix(Ak, degree=n)
            roots_rank, time_rank = companion_solver(fhat)
            roots_rank_ref = newton_refine(f, roots_rank)
            rank_eta = max_backward_error(f, roots_rank_ref)
            rank_coeff_error = coefficient_mismatch(f, fhat)
            if rank_eta > 10 * companion_eta and companion_eta < FAIL_TOL:
                selected_method, selected_eta = "companion fallback", companion_eta
            else:
                selected_method, selected_eta = "rank-aware surrogate", rank_eta
            rows.append({
                "family": family, "degree_n": int(n), "rho_sigma_2": rho_sigma_k(A, 2),
                "rho_sigma_k": rho_sigma_k(A, k), "k": int(k),
                "companion_eta": companion_eta, "rank_eta": rank_eta,
                "selected_eta": selected_eta, "selected_method": selected_method,
                "companion_runtime_s": time_comp, "rank_runtime_s": time_sketch + time_rank,
                "rank_coeff_error": rank_coeff_error,
            })
    return pd.DataFrame(rows)


def experiment_7_method_comparison(degrees=(20, 50, 100, 200), seed=7):
    rng = np.random.default_rng(seed)
    rows = []
    methods = {"companion": companion_solver, "durand_kerner": durand_kerner, "aberth_ehrlich": aberth_ehrlich}
    for n in degrees:
        f, A, v1, v2 = generate_exact_rank2_polynomial(n, rng)
        start = time.perf_counter()
        roots_rank = exact_rank2_roots(v1, v2)[:n]
        roots_rank_ref = newton_refine(f, roots_rank)
        runtime_rank = time.perf_counter() - start
        eta_rank = max_backward_error(f, roots_rank_ref)
        rows.append({
            "degree_n": int(n), "method": "rank-aware exact", "eta_max": eta_rank,
            "normalized_residual": normalized_root_residual(f, roots_rank_ref),
            "runtime_s": runtime_rank, "all_roots": True, "requires_initial_guess": False,
            "status": "ok" if np.isfinite(eta_rank) and eta_rank <= FAIL_TOL else "unstable_or_failed",
        })
        for method_name, solver in methods.items():
            try:
                roots, runtime = solver(f)
                roots_ref = newton_refine(f, roots)
                eta = max_backward_error(f, roots_ref)
                status = "ok" if np.isfinite(eta) and eta <= FAIL_TOL and len(roots) else "unstable_or_failed"
                rows.append({
                    "degree_n": int(n), "method": method_name, "eta_max": eta,
                    "normalized_residual": normalized_root_residual(f, roots_ref), "runtime_s": runtime,
                    "all_roots": True, "requires_initial_guess": method_name in ["durand_kerner", "aberth_ehrlich"],
                    "status": status,
                })
            except Exception as e:
                rows.append({
                    "degree_n": int(n), "method": method_name, "eta_max": np.inf,
                    "normalized_residual": np.inf, "runtime_s": np.nan, "all_roots": True,
                    "requires_initial_guess": method_name in ["durand_kerner", "aberth_ehrlich"],
                    "status": f"failed: {str(e)}",
                })
    return pd.DataFrame(rows)


def eigenpair_residual_linear(M, eigvals, eigvecs):
    Mnorm = np.linalg.norm(M, 2)
    vals = []
    for i, lam in enumerate(eigvals):
        z = eigvecs[:, i]
        num = np.linalg.norm((M - lam * np.eye(M.shape[0])) @ z)
        den = (Mnorm + abs(lam)) * max(np.linalg.norm(z), EPS)
        vals.append(num / max(den, EPS))
    return float(np.max(vals))


def experiment_8_matrix_pencils(sizes=(50, 100, 200, 500, 1000), seed=8):
    rng = np.random.default_rng(seed)
    rows = []
    for m in sizes:
        for kind in ["symmetric", "nonnormal"]:
            M = rng.normal(size=(m, m))
            if kind == "symmetric":
                M = (M + M.T) / 2.0
            start = time.perf_counter()
            eigvals, eigvecs = np.linalg.eig(M)
            runtime = time.perf_counter() - start
            res = eigenpair_residual_linear(M, eigvals, eigvecs)
            rows.append({
                "problem": f"linear_{kind}", "dimension_m": int(m), "pencil_type": "linear",
                "max_r_eig": res, "log10_max_r_eig": np.log10(max(res, EPS)), "runtime_s": runtime,
            })
    for m in [20, 30, 50]:
        K = rng.normal(size=(m, m))
        C = rng.normal(size=(m, m))
        M = np.eye(m)
        Z = np.zeros((m, m))
        I = np.eye(m)
        L = np.block([[Z, I], [-K, -C]])
        start = time.perf_counter()
        eigvals, eigvecs = np.linalg.eig(L)
        runtime = time.perf_counter() - start
        residuals = []
        for idx, lam in enumerate(eigvals):
            z = eigvecs[:m, idx]
            Pz = (K + lam * C + (lam ** 2) * M) @ z
            den = (np.linalg.norm(K, 2) + abs(lam) * np.linalg.norm(C, 2) + abs(lam) ** 2 * np.linalg.norm(M, 2)) * max(np.linalg.norm(z), EPS)
            residuals.append(np.linalg.norm(Pz) / max(den, EPS))
        res = float(np.max(residuals))
        rows.append({
            "problem": "quadratic_random", "dimension_m": int(m), "pencil_type": "quadratic",
            "max_r_eig": res, "log10_max_r_eig": np.log10(max(res, EPS)), "runtime_s": runtime,
        })
    return pd.DataFrame(rows)


def experiment_9_failure_and_ablation(degree=100, seed=9):
    rng = np.random.default_rng(seed)
    rows = []
    f = rng.normal(size=degree + 1)
    f = f / max(np.linalg.norm(f), EPS)
    A = canonical_A_from_poly(f)
    roots_comp, t_comp = companion_solver(f)
    roots_comp_ref = newton_refine(f, roots_comp)
    comp_eta = max_backward_error(f, roots_comp_ref)
    k = 5
    configs = []
    Ak, t_sketch = sketch_qr_rank_k(A, k, oversample=10, seed=seed)
    configs.append(("full framework", k, Ak, t_sketch, "rank-k surrogate"))
    configs.append(("without Newton refinement", k, Ak, t_sketch, "rank-k surrogate_no_ref"))
    start = time.perf_counter()
    Afull = spectral_rank_k(A, k)
    configs.append(("full spectral truncation", k, Afull, time.perf_counter() - start, "spectral surrogate"))
    for bad_k in [1, 2]:
        Abad, t_bad = sketch_qr_rank_k(A, bad_k, oversample=5, seed=seed + bad_k)
        configs.append((f"failure small rank k={bad_k}", bad_k, Abad, t_bad, "low-rank surrogate"))
    for name, kk, Amodel, t_model, method_label in configs:
        fhat = anti_diagonal_poly_from_matrix(Amodel, degree=degree)
        roots, t_root = companion_solver(fhat)
        roots_eval = roots if name == "without Newton refinement" else newton_refine(f, roots)
        eta = max_backward_error(f, roots_eval)
        fallback = eta > 10 * comp_eta and comp_eta < FAIL_TOL
        rows.append({
            "configuration": name, "k": int(kk), "rel_coeff_error": coefficient_mismatch(f, fhat),
            "eta_max": eta, "selected_eta": comp_eta if fallback else eta,
            "selected_method": "companion fallback" if fallback else method_label,
            "runtime_s": t_model + t_root, "rho_sigma_k": rho_sigma_k(A, kk),
        })
    rows.append({
        "configuration": "companion fallback", "k": np.nan, "rel_coeff_error": 0.0,
        "eta_max": comp_eta, "selected_eta": comp_eta, "selected_method": "companion",
        "runtime_s": t_comp, "rho_sigma_k": np.nan,
    })
    return pd.DataFrame(rows)


def bootstrap_ci(values, n_boot=2000, ci=95, seed=10):
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    boots = [np.mean(rng.choice(values, size=len(values), replace=True)) for _ in range(n_boot)]
    alpha = (100 - ci) / 2
    return float(np.percentile(boots, alpha)), float(np.percentile(boots, 100 - alpha))


def experiment_10_statistical_perturbation(degree=50, noise_levels=(1e-12, 1e-10, 1e-8, 1e-6, 1e-4), trials=100, seed=10):
    rng = np.random.default_rng(seed)
    rows = []
    raw_rows = []
    for eps in noise_levels:
        eta_clean_list, eta_pert_list, drift_list, rho_list = [], [], [], []
        for t in range(trials):
            f, A, v1, v2 = generate_exact_rank2_polynomial(degree, rng)
            roots_clean = exact_rank2_roots(v1, v2)[:degree]
            roots_clean_ref = newton_refine(f, roots_clean)
            eta_clean = max_backward_error(f, roots_clean_ref)
            E = rng.normal(size=A.shape)
            E = (E + E.T) / 2.0
            E = E / max(np.linalg.norm(E, 2), EPS)
            A_eps = A + eps * np.linalg.norm(A, 2) * E
            f_eps = anti_diagonal_poly_from_matrix(A_eps, degree=degree)
            f_eps = f_eps / max(np.linalg.norm(f_eps), EPS)
            roots_pert, _ = companion_solver(f_eps)
            roots_pert_ref = newton_refine(f_eps, roots_pert)
            eta_pert = max_backward_error(f_eps, roots_pert_ref)
            drift = np.nan
            if len(roots_pert_ref) == len(roots_clean_ref):
                drift = float(np.median(np.abs(np.sort_complex(roots_pert_ref) - np.sort_complex(roots_clean_ref))))
            rho = rho_sigma_k(A_eps, 2)
            eta_clean_list.append(eta_clean); eta_pert_list.append(eta_pert); drift_list.append(drift); rho_list.append(rho)
            raw_rows.append({"epsilon": float(eps), "trial": int(t), "eta_clean": eta_clean, "eta_perturbed": eta_pert, "root_drift": drift, "rho_sigma_2": rho})
        eta_clean_arr = np.asarray(eta_clean_list, dtype=float)
        eta_pert_arr = np.asarray(eta_pert_list, dtype=float)
        finite_pert = eta_pert_arr[np.isfinite(eta_pert_arr)]
        ci_low, ci_high = bootstrap_ci(finite_pert, seed=seed)
        valid_pair = np.isfinite(eta_clean_arr) & np.isfinite(eta_pert_arr)
        if stats is not None and np.sum(valid_pair) > 2:
            t_stat, p_value = stats.ttest_rel(eta_clean_arr[valid_pair], eta_pert_arr[valid_pair])
        else:
            t_stat, p_value = np.nan, np.nan
        rows.append({
            "degree_n": int(degree), "epsilon": float(eps), "trials": int(trials),
            "eta_mean": float(np.mean(finite_pert)) if finite_pert.size else np.inf,
            "eta_sd": float(np.std(finite_pert, ddof=1)) if finite_pert.size > 1 else np.nan,
            "eta_median": float(np.median(finite_pert)) if finite_pert.size else np.inf,
            "eta_boot95_low": ci_low, "eta_boot95_high": ci_high,
            "rho_mean": float(np.nanmean(rho_list)), "rho_sd": float(np.nanstd(rho_list, ddof=1)),
            "root_drift_mean": float(np.nanmean(drift_list)), "root_drift_sd": float(np.nanstd(drift_list, ddof=1)),
            "paired_t_stat": float(t_stat) if np.isfinite(t_stat) else np.nan,
            "paired_p_value": float(p_value) if np.isfinite(p_value) else np.nan,
            "failure_rate": float(np.mean(~np.isfinite(eta_pert_arr) | (eta_pert_arr > FAIL_TOL))),
        })
    return pd.DataFrame(rows), pd.DataFrame(raw_rows)


# ============================================================
# Export and figures
# ============================================================

def save_table(df, name):
    csv_path = os.path.join(OUTPUT_DIR, f"{name}.csv")
    tex_path = os.path.join(OUTPUT_DIR, f"{name}.tex")
    df.to_csv(csv_path, index=False)
    with open(tex_path, "w", encoding="utf-8") as f:
        f.write(df.to_latex(index=False, escape=False, float_format=lambda x: f"{x:.3e}"))
    return csv_path, tex_path


def save_fig(name):
    path_pdf = os.path.join(FIG_DIR, f"{name}.pdf")
    path_png = os.path.join(FIG_DIR, f"{name}.png")
    plt.tight_layout()
    plt.savefig(path_pdf, bbox_inches="tight")
    plt.savefig(path_png, dpi=300, bbox_inches="tight")
    plt.close()
    return path_pdf


def generate_all_figures(experiments):
    # Exp 1
    df = experiments["exp1_exact_rank2"]
    summary = df.groupby("degree_n")[["sigma3_A", "Delta3_A", "eta_max_refined"]].median()
    plt.figure(figsize=(6, 4))
    plt.semilogy(summary.index, summary["sigma3_A"], marker="o", label=r"$\sigma_3(A)$")
    plt.semilogy(summary.index, summary["eta_max_refined"], marker="s", label=r"$\eta_{max}^{ref}$")
    plt.xlabel("Polynomial degree"); plt.ylabel("Median diagnostic"); plt.title("Exact rank--2 validation"); plt.grid(True, which="both", linestyle="--", linewidth=0.5); plt.legend(); save_fig("fig_exp1_exact_rank2")

    # Exp 2
    df = experiments["exp2_statistical_robustness"]
    plt.figure(figsize=(6, 4))
    plt.errorbar(df["degree_n"], df["eta_mean"], yerr=df["eta_sd"], marker="o", capsize=4)
    plt.yscale("log"); plt.xlabel("Polynomial degree"); plt.ylabel(r"Backward error $\eta_{max}$"); plt.title("Statistical robustness"); plt.grid(True, which="both", linestyle="--", linewidth=0.5); save_fig("fig_exp2_statistical_robustness")

    # Exp 3
    df = experiments["exp3_perturbation"]
    summary = df.groupby("epsilon")[["eta_max_refined", "rho_sigma_2", "root_drift_median"]].median()
    plt.figure(figsize=(6, 4))
    plt.loglog(summary.index, summary["eta_max_refined"], marker="o", label=r"$\eta_{max}^{ref}$")
    plt.loglog(summary.index, summary["rho_sigma_2"], marker="s", label=r"$\rho_{\sigma,2}$")
    plt.xlabel(r"Noise level $\epsilon$"); plt.ylabel("Median value"); plt.title("Perturbation stability"); plt.grid(True, which="both", linestyle="--", linewidth=0.5); plt.legend(); save_fig("fig_exp3_perturbation_stability")

    # Exp 4
    df = experiments["exp4_rankk_surrogate"]
    plt.figure(figsize=(6, 4))
    for degree, group in df.groupby("degree_n"):
        group = group.sort_values("k")
        plt.semilogy(group["k"], group["rel_coeff_error"], marker="o", label=f"n={degree}")
    plt.xlabel("Surrogate rank k"); plt.ylabel(r"$\|f-\hat f_k\|/\|f\|$"); plt.title("Rank--k surrogate quality"); plt.grid(True, which="both", linestyle="--", linewidth=0.5); plt.legend(); save_fig("fig_exp4_rankk_coeff_error")

    # Exp 5
    df = experiments["exp5_sketch_qr_scalability"]
    plt.figure(figsize=(6, 4))
    plt.loglog(df["degree_n"], df["full_eig_time_s"], marker="o", label="Full eig.")
    plt.loglog(df["degree_n"], df["sketch_qr_time_s"], marker="s", label="Sketch--QR")
    plt.xlabel("Polynomial degree"); plt.ylabel("Runtime (s)"); plt.title("Runtime scaling"); plt.grid(True, which="both", linestyle="--", linewidth=0.5); plt.legend(); save_fig("fig_exp5_runtime_scaling")

    # Exp 6
    df = experiments["exp6_benchmark_polynomials"].dropna(subset=["companion_eta", "rank_eta"])
    plt.figure(figsize=(6, 4))
    for family, group in df.groupby("family"):
        group = group.sort_values("degree_n")
        plt.semilogy(group["degree_n"], group["companion_eta"], marker="o", label=f"{family} companion")
        plt.semilogy(group["degree_n"], group["rank_eta"], marker="s", label=f"{family} rank-aware")
    plt.xlabel("Polynomial degree"); plt.ylabel(r"Backward error $\eta_{max}$"); plt.title("Benchmark comparison"); plt.grid(True, which="both", linestyle="--", linewidth=0.5); plt.legend(fontsize=8); save_fig("fig_exp6_benchmark_comparison")

    # Exp 7
    df = experiments["exp7_method_comparison"].dropna(subset=["eta_max"])
    pivot = df.groupby("method")["eta_max"].median().sort_values()
    plt.figure(figsize=(7, 4))
    plt.bar(range(len(pivot)), pivot.values)
    plt.yscale("log"); plt.xticks(range(len(pivot)), pivot.index, rotation=30, ha="right"); plt.ylabel(r"Median $\eta_{max}$"); plt.title("Rootfinding methods"); plt.grid(True, axis="y", which="both", linestyle="--", linewidth=0.5); save_fig("fig_exp7_method_comparison")

    # Exp 8
    df = experiments["exp8_matrix_pencils"].dropna(subset=["max_r_eig"])
    plt.figure(figsize=(6, 4))
    for problem, group in df.groupby("problem"):
        group = group.sort_values("dimension_m")
        plt.semilogy(group["dimension_m"], group["max_r_eig"], marker="o", label=problem)
    plt.xlabel("Matrix dimension"); plt.ylabel(r"Eigenpair residual $r_{eig}$"); plt.title("Matrix-pencil validation"); plt.grid(True, which="both", linestyle="--", linewidth=0.5); plt.legend(fontsize=8); save_fig("fig_exp8_matrix_pencils")

    # Exp 9
    df = experiments["exp9_failure_ablation"].dropna(subset=["eta_max"])
    plt.figure(figsize=(7, 4))
    plt.bar(range(len(df)), df["eta_max"])
    plt.yscale("log"); plt.xticks(range(len(df)), df["configuration"], rotation=30, ha="right"); plt.ylabel(r"$\eta_{max}$"); plt.title("Ablation and failure regimes"); plt.grid(True, axis="y", which="both", linestyle="--", linewidth=0.5); save_fig("fig_exp9_ablation")

    # Exp 10 boxplot
    df = experiments["exp10_statistical_perturbation_raw"].copy()
    df["epsilon_label"] = df["epsilon"].map(lambda x: f"{x:.0e}")
    plt.figure(figsize=(7, 4))
    df.boxplot(column="eta_perturbed", by="epsilon_label", grid=False)
    plt.yscale("log"); plt.xlabel(r"Perturbation level $\epsilon$"); plt.ylabel(r"Backward error $\eta_{max}$"); plt.title("Statistical perturbation validation"); plt.suptitle(""); save_fig("fig_exp10_perturbation_boxplot")

    # Singular value decay
    rng = np.random.default_rng(123)
    plt.figure(figsize=(6, 4))
    for n in [50, 100, 200]:
        _, A, _, _ = generate_exact_rank2_polynomial(n, rng)
        s = singular_values(A); s = s / max(s[0], EPS)
        plt.semilogy(np.arange(1, len(s) + 1), s, marker="o", label=f"n={n}")
    plt.xlabel("Singular-value index"); plt.ylabel(r"$\sigma_i/\sigma_1$"); plt.title("Singular-value decay"); plt.grid(True, which="both", linestyle="--", linewidth=0.5); plt.legend(); save_fig("fig_singular_value_decay")

    print(f"\nFigures saved in: {FIG_DIR}")


def run_all_experiments():
    warnings.filterwarnings("ignore")
    experiments: Dict[str, pd.DataFrame] = {}
    experiments["exp1_exact_rank2"] = experiment_1_exact_rank2()
    experiments["exp2_statistical_robustness"] = experiment_2_statistical_robustness()
    experiments["exp3_perturbation"] = experiment_3_perturbation()
    experiments["exp4_rankk_surrogate"] = experiment_4_rankk_surrogate()
    experiments["exp5_sketch_qr_scalability"] = experiment_5_sketch_qr_scalability()
    experiments["exp6_benchmark_polynomials"] = experiment_6_benchmark_polynomials()
    experiments["exp7_method_comparison"] = experiment_7_method_comparison()
    experiments["exp8_matrix_pencils"] = experiment_8_matrix_pencils()
    experiments["exp9_failure_ablation"] = experiment_9_failure_and_ablation()
    exp10_summary, exp10_raw = experiment_10_statistical_perturbation()
    experiments["exp10_statistical_perturbation"] = exp10_summary
    experiments["exp10_statistical_perturbation_raw"] = exp10_raw

    xlsx_path = os.path.join(OUTPUT_DIR, "all_10_experiments.xlsx")
    with pd.ExcelWriter(xlsx_path) as writer:
        for name, df in experiments.items():
            save_table(df, name)
            df.to_excel(writer, sheet_name=name[:31], index=False)

    print("\nAll 10 experiments completed.")
    print(f"Results saved in: {OUTPUT_DIR}")
    print(f"Excel workbook: {xlsx_path}")
    for name, df in experiments.items():
        print("\n" + "=" * 80)
        print(name)
        print("=" * 80)
        print(df.head(10).to_string(index=False))
    return experiments


if __name__ == "__main__":
    experiments = run_all_experiments()
    generate_all_figures(experiments)


All 10 experiments completed.
Results saved in: rank_aware_10_experiment_outputs
Excel workbook: rank_aware_10_experiment_outputs/all_10_experiments.xlsx

exp1_exact_rank2
 degree_n  trial  m  rank_A     sigma3_A     Delta3_A      eta_max  eta_max_refined  normalized_root_residual  raw_root_residual_appendix
        5      0  4       2 2.270180e-16 7.815020e-17 4.345578e-17     6.316906e-18              6.316906e-18                7.494005e-15
        5      1  4       2 3.052994e-17 4.273468e-18 9.418522e-17     2.506710e-17              2.506710e-17                3.700880e-16
        5      2  4       2 9.264327e-17 3.916659e-17 2.123889e-17     1.274939e-17              1.274939e-17                7.159667e-11
        5      3  4       2 7.517917e-16 2.660008e-14 5.423654e-17     3.665049e-18              3.665049e-18                9.714451e-17
        5      4  4       2 1.790126e-16 1.633839e-16 2.007467e-16     1.368728e-17              1.368728e-17                7.494005e-16

<Figure size 700x400 with 0 Axes>